# Hanoi Air Live Prototype

Prototype flow: load hybrid live/sample data, train a short LSTM smoke model, run plume-aware 24h forecast, and visualize Bắc Từ Liêm.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from hanoi_air.config import get_settings
from hanoi_air.ingestion import load_air_readings, load_json, parse_aqicn_payload
from hanoi_air.model import LSTMForecaster, make_supervised_sequences, synthetic_training_matrix
from hanoi_air.forecast import build_forecast, top_n_worst
from hanoi_air.viz import build_folium_map

settings = get_settings()
settings

In [ ]:
readings = load_air_readings(settings, use_live=False)
len(readings), readings[:3]

In [ ]:
sample_payload = load_json(settings.sample_dir / 'aqicn_sample.json')
parse_aqicn_payload(sample_payload)

In [ ]:
history = synthetic_training_matrix(length=120, base=45.0)
x, y = make_supervised_sequences(history, lookback=24, horizon=24)
model = LSTMForecaster(input_size=x.shape[-1], hidden_size=12, horizon=24)
model.fit(x[:48], y[:48], epochs=2)
pred = model.predict(x[-1])
pred[:5], pred.shape

In [ ]:
bundle = build_forecast(settings, use_live=False)
bundle['mode'], bundle['district_count'], bundle['max_aqi'], len(bundle['alerts'])

In [ ]:
top_n_worst(bundle, 5)

In [ ]:
m = build_folium_map(bundle, hour_offset=6, selected_district='bac_tu_liem')
m